# Pairs Trading Strategy Demonstration

This notebook recreates `qc_projects/main_pairs_trading.py`, which: 
- Trades the spread ratio between SPY and QQQ
- Enters when spread Z-score exceeds ±2.0
- Exits when Z-score mean reverts inside ±0.5
- Uses market-neutral 50% / -50% pair positioning

## Step 1: Imports and Parameters

We mirror the Lean setup from 2020-01-01 to 2022-01-01 with a 20-day rolling spread window and 25-bar warmup.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option('display.max_columns', None)

spy_symbol = 'SPY'
qqq_symbol = 'QQQ'
start_date = '2020-01-01'
end_date = '2022-01-01'
window = 20
entry_threshold = 2.0
exit_threshold = 0.5
position_size = 0.5
warmup_period = window + 5

print(f'Pair: {spy_symbol}/{qqq_symbol}')
print(f'Backtest window: {start_date} → {end_date}')
print(f'Window={window}, Entry=±{entry_threshold}, Exit=±{exit_threshold}, Position={position_size:.1f}')

## Step 2: Download Daily OHLCV

We request daily bars for both ETFs and align them on the same trading dates.

In [ ]:
spy_df = yf.download(
    spy_symbol,
    start=start_date,
    end=end_date,
    interval='1d',
    progress=False,
    multi_level_index=False,
    auto_adjust=False,
)
qqq_df = yf.download(
    qqq_symbol,
    start=start_date,
    end=end_date,
    interval='1d',
    progress=False,
    multi_level_index=False,
    auto_adjust=False,
)

if spy_df.empty or qqq_df.empty:
    raise RuntimeError('Missing OHLCV data for one or both symbols.')

pair_df = pd.DataFrame(index=spy_df.index.union(qqq_df.index).sort_values())
pair_df['spy_close'] = spy_df['Close']
pair_df['qqq_close'] = qqq_df['Close']
pair_df['spy_open'] = spy_df['Open']
pair_df['qqq_open'] = qqq_df['Open']
pair_df = pair_df.dropna().copy()
pair_df.index = pd.to_datetime(pair_df.index).tz_localize(None)
pair_df.index.name = 'date'

print(f'Aligned daily rows: {len(pair_df):,}')
pair_df.head()

## Step 3: Build Spread, Rolling Stats, and Z-Score

This is the same core formula from Lean: spread = SPY/QQQ and z-score over a rolling 20-day window.

In [ ]:
pair_df['spread'] = pair_df['spy_close'] / pair_df['qqq_close']
pair_df['mean_spread'] = pair_df['spread'].rolling(window).mean()
pair_df['std_spread'] = pair_df['spread'].rolling(window).std(ddof=0)
pair_df['z_score'] = (pair_df['spread'] - pair_df['mean_spread']) / pair_df['std_spread']

pair_df['z_score'] = pair_df['z_score'].replace([np.inf, -np.inf], np.nan)
pair_df = pair_df.dropna(subset=['z_score']).copy()
pair_df = pair_df.iloc[warmup_period:] if len(pair_df) > warmup_period else pair_df

pair_df[['spread', 'mean_spread', 'std_spread', 'z_score']].head()

## Step 4: Replay Lean Entry/Exit State Machine

- Enter short SPY / long QQQ when z-score > +2.0
- Enter long SPY / short QQQ when z-score < -2.0
- Exit both legs when |z-score| < 0.5

In [ ]:
trades = []
entry_markers = []
exit_markers = []
active_position = None

for ts, row in pair_df.iterrows():
    z = float(row['z_score'])
    spy_px = float(row['spy_close'])
    qqq_px = float(row['qqq_close'])

    if active_position is None:
        if z > entry_threshold:
            active_position = {
                'side': 'short_spy_long_qqq',
                'entry_time': ts,
                'entry_z': z,
                'spy_entry': spy_px,
                'qqq_entry': qqq_px,
            }
            entry_markers.append({'timestamp': ts, 'spread': row['spread'], 'z_score': z})
        elif z < -entry_threshold:
            active_position = {
                'side': 'long_spy_short_qqq',
                'entry_time': ts,
                'entry_z': z,
                'spy_entry': spy_px,
                'qqq_entry': qqq_px,
            }
            entry_markers.append({'timestamp': ts, 'spread': row['spread'], 'z_score': z})
    else:
        if abs(z) < exit_threshold:
            spy_ret = (spy_px / active_position['spy_entry']) - 1.0
            qqq_ret = (qqq_px / active_position['qqq_entry']) - 1.0

            if active_position['side'] == 'short_spy_long_qqq':
                pnl = (-position_size * spy_ret) + (position_size * qqq_ret)
            else:
                pnl = (position_size * spy_ret) + (-position_size * qqq_ret)

            trades.append({
                'side': active_position['side'],
                'entry_time': active_position['entry_time'],
                'exit_time': ts,
                'entry_z': round(active_position['entry_z'], 3),
                'exit_z': round(z, 3),
                'spy_entry': round(active_position['spy_entry'], 2),
                'spy_exit': round(spy_px, 2),
                'qqq_entry': round(active_position['qqq_entry'], 2),
                'qqq_exit': round(qqq_px, 2),
                'return_pct': round(pnl * 100, 3),
            })
            exit_markers.append({'timestamp': ts, 'spread': row['spread'], 'z_score': z})
            active_position = None

if active_position is not None:
    last_ts = pair_df.index[-1]
    last_row = pair_df.iloc[-1]
    spy_ret = (last_row['spy_close'] / active_position['spy_entry']) - 1.0
    qqq_ret = (last_row['qqq_close'] / active_position['qqq_entry']) - 1.0
    if active_position['side'] == 'short_spy_long_qqq':
        pnl = (-position_size * spy_ret) + (position_size * qqq_ret)
    else:
        pnl = (position_size * spy_ret) + (-position_size * qqq_ret)

    trades.append({
        'side': active_position['side'],
        'entry_time': active_position['entry_time'],
        'exit_time': last_ts,
        'entry_z': round(active_position['entry_z'], 3),
        'exit_z': round(float(last_row['z_score']), 3),
        'spy_entry': round(active_position['spy_entry'], 2),
        'spy_exit': round(float(last_row['spy_close']), 2),
        'qqq_entry': round(active_position['qqq_entry'], 2),
        'qqq_exit': round(float(last_row['qqq_close']), 2),
        'return_pct': round(pnl * 100, 3),
    })
    exit_markers.append({'timestamp': last_ts, 'spread': float(last_row['spread']), 'z_score': float(last_row['z_score'])})

trades_df = pd.DataFrame(trades)
print(f'Total closed trades: {len(trades_df)}')
trades_df.head() if not trades_df.empty else 'No trades generated.'

## Step 5: Performance Summary

We compute trade-level stats to compare threshold settings before updating the QC algorithm.

In [ ]:
if trades_df.empty:
    summary = {
        'total_trades': 0,
        'win_rate_pct': 0.0,
        'avg_return_pct': 0.0,
        'median_return_pct': 0.0,
        'total_return_pct_sum': 0.0,
    }
else:
    summary = {
        'total_trades': int(len(trades_df)),
        'win_rate_pct': round((trades_df['return_pct'] > 0).mean() * 100, 1),
        'avg_return_pct': round(float(trades_df['return_pct'].mean()), 3),
        'median_return_pct': round(float(trades_df['return_pct'].median()), 3),
        'total_return_pct_sum': round(float(trades_df['return_pct'].sum()), 3),
    }

summary

## Step 6: Visualize Spread and Z-Score Signals

We chart spread ratio and z-score thresholds with entry/exit markers, then save it to `notebooks/graphs/`.

In [ ]:
graphs_dir = Path('graphs')
graphs_dir.mkdir(exist_ok=True)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.55, 0.45],
    subplot_titles=('SPY/QQQ Spread', 'Spread Z-Score'),
)

fig.add_trace(
    go.Scatter(x=pair_df.index, y=pair_df['spread'], name='Spread (SPY/QQQ)', line=dict(color='#1f77b4')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=pair_df.index, y=pair_df['mean_spread'], name=f'Rolling Mean ({window})', line=dict(color='#ff7f0e', dash='dash')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=pair_df.index, y=pair_df['z_score'], name='Z-Score', line=dict(color='#9467bd')),
    row=2, col=1
)
fig.add_hline(y=entry_threshold, line=dict(color='#d62728', dash='dash'), row=2, col=1)
fig.add_hline(y=-entry_threshold, line=dict(color='#d62728', dash='dash'), row=2, col=1)
fig.add_hline(y=exit_threshold, line=dict(color='#2ca02c', dash='dot'), row=2, col=1)
fig.add_hline(y=-exit_threshold, line=dict(color='#2ca02c', dash='dot'), row=2, col=1)
fig.add_hline(y=0, line=dict(color='#7f7f7f', dash='dot'), row=2, col=1)

if entry_markers:
    fig.add_trace(
        go.Scatter(
            x=[m['timestamp'] for m in entry_markers],
            y=[m['z_score'] for m in entry_markers],
            mode='markers',
            marker=dict(symbol='triangle-up', size=9, color='#2ca02c'),
            name='Entries',
        ),
        row=2, col=1
    )
if exit_markers:
    fig.add_trace(
        go.Scatter(
            x=[m['timestamp'] for m in exit_markers],
            y=[m['z_score'] for m in exit_markers],
            mode='markers',
            marker=dict(symbol='triangle-down', size=9, color='#d62728'),
            name='Exits',
        ),
        row=2, col=1
    )

fig.update_layout(height=760, title='Pairs Trading (SPY/QQQ) Spread and Z-Score Signals')
output_path = graphs_dir / 'pairs_trading_signals.html'
fig.write_html(output_path)
fig.show()
print(f'Saved interactive chart to {output_path}')

## Step 7: Inspect Trades and Quick Recommendation

This mirrors the report-style conclusion used by the other notebooks.

In [ ]:
if not trades_df.empty:
    display(trades_df.tail(10))

if summary['total_trades'] == 0:
    recommendation = 'HOLD - no qualifying pair divergence under current thresholds.'
elif summary['win_rate_pct'] >= 55 and summary['avg_return_pct'] > 0:
    recommendation = 'BUY (pairs strategy active) - mean reversion behavior appears favorable.'
elif summary['avg_return_pct'] <= 0:
    recommendation = 'AVOID/REFINE - current entry/exit thresholds are not producing positive expectancy.'
else:
    recommendation = 'NEUTRAL - mixed outcomes; consider tuning window and z-score thresholds.'

print('Recommendation:', recommendation)

## Recap

- This notebook reproduces the same state machine from `main_pairs_trading.py` using Yahoo daily bars.
- Use this local run to tune `window`, `entry_threshold`, and `exit_threshold` before syncing back to QuantConnect.
- The saved artifact `graphs/pairs_trading_signals.html` can be reused in docs and report workflows.